# 01 — Exploratory Data Analysis

**Business question:** How serious is churn, and which customer groups contribute most to it?

This notebook supports the analyst workflow:
1. Load IBM Telco data if available; otherwise use generated demo data.
2. Clean and engineer business features.
3. Measure churn KPIs.
4. Identify high-churn segments.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.features import clean_telco, add_business_features

ibm_path = ROOT / "data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
demo_path = ROOT / "data/processed/demo_customers.csv"

if ibm_path.exists():
    df = add_business_features(clean_telco(pd.read_csv(ibm_path)))
    print("Using IBM Telco dataset")
else:
    if not demo_path.exists():
        exec((ROOT/"data/generate_demo_data.py").read_text())
        main()
    df = add_business_features(pd.read_csv(demo_path))
    print("Using synthetic demo dataset")

df.head()

In [ ]:
summary = {
    "customers": len(df),
    "churned": int(df["churn_flag"].sum()),
    "churn_rate": df["churn_flag"].mean()
}
summary

In [ ]:
contract_col = "contract" if "contract" in df.columns else "contract_type"
contract_churn = (
    df.groupby(contract_col, observed=True)
      .agg(customers=("customer_id","count"),
           churn_rate=("churn_flag","mean"),
           avg_monthly_charges=("monthly_charges","mean"))
      .sort_values("churn_rate", ascending=False)
)
contract_churn

In [ ]:
tenure_churn = (
    df.groupby("tenure_group", observed=True)
      .agg(customers=("customer_id","count"),
           churn_rate=("churn_flag","mean"))
)
tenure_churn

In [ ]:
contract_churn["churn_rate"].plot(kind="bar", title="Churn rate by contract type")
plt.ylabel("Churn rate")
plt.tight_layout()
plt.show()

## Business interpretation

Write 3–5 concise findings here. Focus on:
- which segments have the highest churn,
- whether high charges coincide with high churn,
- which segments should be investigated for retention targeting,
- likely commercial implications rather than statistical description alone.